In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ════════════════════════════════════════════════════════════════════════════
# config.py  — all knobs in one place
# ════════════════════════════════════════════════════════════════════════════

CAT_FEATURES = [
    "dealer_id", "region", "tier_city", "zone",
    "model", "variant", "segment", "fuel_type", "transmission",
    "allocation_policy", "credit_rating",
    "dealer_cluster",
]

FEATURES = [
    "dealer_id", "region", "tier_city", "zone",
    "model", "variant", "segment", "fuel_type", "transmission",
    "allocation_policy", "credit_rating", "dealer_cluster",

    "dealership_age_years", "num_sales_staff", "showroom_area_sqft",

    "demand_lag1", "demand_lag3", "demand_lag6", "demand_lag12",
    "demand_roll3", "demand_roll6",
    "demand_trend3", "yoy_demand_growth",

    # ── ADVANCED SEASONALITY (The WAPE Killers) ──
    "demand_roll12", "demand_roll12_std", "yoy_roll_ratio",

    # ── FUNNEL & LOYALTY ──
    "enquiries_lag1", "test_drives_lag1",
    "enquiry_to_testdrive_ratio", "testdrive_to_booking_ratio",
    "repeat_order_pct",

    "bookings_lag1", "booking_roll3",
    "cancel_rate_roll3",
    "conversion_rate_roll6",
    "booking_implied_demand",

    "inv_lag1", "fill_rate_lag1", "coverage_days", "supply_shock_flag",

    "dealer_infl_roll3", "panic_order_lag1",

    "current_price_inr", "discount_offered_inr",
    "price_change_lag1", "campaign_active",
    "months_since_launch", "new_variant_flag",
    "competitor_segment_launch_lag1",

    "fuel_price_inr_per_litre",
    "festive_month_flag", "holiday_count",
    "regional_gdp_growth_pct", "rainfall_mm",

    "sin_month_k1", "cos_month_k1",
    "sin_month_k2", "cos_month_k2",
    "sin_month_k3", "cos_month_k3",
    "year",
]

HORIZONS = [1, 3, 6]

STOCKOUT_COST_RATIO = 0.30
HOLDING_COST_RATIO  = 0.05

SAFETY_MULTIPLIERS = {1: 1.18, 3: 1.12, 6: 1.08}
MIN_HISTORY_MONTHS = 12
VALIDATION_WINDOWS = 6


# ════════════════════════════════════════════════════════════════════════════
# load.py
# ════════════════════════════════════════════════════════════════════════════

def load_and_merge(
    dealer_fact_path   = "fact_dealer_monthly.csv",
    dealer_dim_path    = "dim_dealers.csv",
    model_dim_path     = "dim_models.csv",
    model_monthly_path = "fact_model_monthly.csv",
    external_path      = "fact_external_monthly.csv",
    production_path    = "fact_production_monthly.csv",
) -> pd.DataFrame:
    dealer_fact = pd.read_csv(dealer_fact_path)
    dealer_dim  = pd.read_csv(dealer_dim_path)
    model_dim   = pd.read_csv(model_dim_path)
    model_mnth  = pd.read_csv(model_monthly_path)
    external    = pd.read_csv(external_path)
    production  = pd.read_csv(production_path)

    dealer_dim = dealer_dim.drop(columns=["allocation_policy"], errors="ignore")

    df = dealer_fact.merge(
        dealer_dim[[
            "dealer_id", "region", "city", "tier_city", "zone",
            "dealership_age_years", "showroom_area_sqft",
            "service_center", "num_sales_staff", "credit_rating",
        ]], on="dealer_id", how="left"
    ).merge(
        model_mnth.drop(columns=["year", "month"], errors="ignore"), on=["model", "variant", "year_month"], how="left"
    ).merge(
        external.drop(columns=["year", "month"], errors="ignore"), on=["region", "year_month"], how="left"
    ).merge(
        production[[
            "model", "variant", "year_month",
            "monthly_capacity", "planned_production",
            "actual_production", "supply_shock_flag",
        ]], on=["model", "variant", "year_month"], how="left"
    ).merge(
        model_dim[["model", "variant", "segment", "fuel_type", "transmission"]], on=["model", "variant"], how="left"
    )

    return df.sort_values(["dealer_id", "model", "variant", "year", "month"]).reset_index(drop=True)


# ════════════════════════════════════════════════════════════════════════════
# demand.py & features.py
# ════════════════════════════════════════════════════════════════════════════

def reconstruct_true_demand(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(["dealer_id", "model", "variant"])

    df["true_demand"] = (df["retail_sales"] + df["opening_inventory"] - df["closing_inventory"]).clip(lower=0)

    df["demand_unbiased"] = np.where(df["stockout_flag"] == 1, np.nan, df["true_demand"])
    df["true_demand"] = g["demand_unbiased"].transform(lambda s: s.fillna(s.rolling(3, min_periods=1).mean())).fillna(df["true_demand"])

    df["net_bookings"] = (df["bookings"] - df["cancellations"]).clip(lower=0)
    df["cancel_rate"]  = df["cancellations"] / (df["bookings"] + 1)
    df["dealer_inflation"] = df["dealer_requested_qty"] - df["true_demand"]
    df["dealer_infl_roll3"] = g["dealer_inflation"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    df["panic_order_lag1"] = g["panic_order_flag"].shift(1).fillna(0)

    return df


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(["dealer_id", "model", "variant"])

    for lag in [1, 3, 6, 12]:
        df[f"demand_lag{lag}"] = g["true_demand"].shift(lag)

    for window in [3, 6]:
        df[f"demand_roll{window}"] = g["true_demand"].shift(1).transform(lambda s: s.rolling(window, min_periods=1).mean())

    # ── Advanced Seasonality Math ──
    df["demand_roll12"] = g["true_demand"].shift(1).transform(lambda s: s.rolling(12, min_periods=6).mean())
    df["demand_roll12_std"] = g["true_demand"].shift(1).transform(lambda s: s.rolling(12, min_periods=6).std()).fillna(0)
    df["yoy_roll_ratio"] = df["demand_roll3"] / (df["demand_roll12"] + 1)

    df["demand_trend3"]     = (df["demand_lag1"] - df["demand_lag3"]) / 2.0
    df["yoy_demand_growth"] = (df["demand_lag1"] - df["demand_lag12"]) / (df["demand_lag12"] + 1)
    df["bookings_lag1"] = g["net_bookings"].shift(1)
    df["booking_roll3"] = g["net_bookings"].shift(1).transform(lambda s: s.rolling(3, min_periods=1).mean())
    df["cancel_rate_roll3"] = g["cancel_rate"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

    df["conversion_rate_roll6"] = (
        g["retail_sales"].shift(1).transform(lambda s: s.rolling(6, min_periods=1).sum()) /
        (g["bookings"].shift(1).transform(lambda s: s.rolling(6, min_periods=1).sum()) + 1)
    )
    df["booking_implied_demand"] = df["bookings_lag1"] * df["conversion_rate_roll6"]

    if "enquiries" in df.columns and "test_drives" in df.columns:
        df["enquiries_lag1"] = g["enquiries"].shift(1)
        df["test_drives_lag1"] = g["test_drives"].shift(1)
        df["enquiry_to_testdrive_ratio"] = g["test_drives"].shift(1) / (g["enquiries"].shift(1) + 1)
        df["testdrive_to_booking_ratio"] = g["bookings"].shift(1) / (g["test_drives"].shift(1) + 1)
        df["repeat_order_pct"] = g["repeat_orders"].shift(1) / (g["retail_sales"].shift(1) + 1)

    df["inv_lag1"]       = g["closing_inventory"].shift(1)
    df["fill_rate_lag1"] = g["allocated_qty"].shift(1) / (g["dealer_requested_qty"].shift(1) + 1)
    df["coverage_days"]  = df["inv_lag1"] / (df["demand_roll3"] / 30.0 + 0.001)

    df["price_change_lag1"] = g["price_change_pct"].shift(1)
    if "competitor_segment_launch" in df.columns:
        df["competitor_segment_launch_lag1"] = g["competitor_segment_launch"].shift(1)

    for k in [1, 2, 3]:
        df[f"sin_month_k{k}"] = np.sin(2 * np.pi * k * df["month"] / 12)
        df[f"cos_month_k{k}"] = np.cos(2 * np.pi * k * df["month"] / 12)

    return df

def assign_dealer_clusters(train_df: pd.DataFrame, test_df: pd.DataFrame, n_clusters=6):
    cluster_feats = ["demand_roll12", "cancel_rate_roll3", "dealer_infl_roll3", "fill_rate_lag1"]
    avail = [f for f in cluster_feats if f in train_df.columns]

    if not avail:
        train_df["dealer_cluster"] = "unknown"
        test_df["dealer_cluster"]  = "unknown"
        return train_df, test_df

    profiles = train_df.groupby("dealer_id")[avail].mean().fillna(0).reset_index()
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    profiles["dealer_cluster"] = km.fit_predict(profiles[avail]).astype(str)

    cmap = profiles.set_index("dealer_id")["dealer_cluster"].to_dict()
    train_df["dealer_cluster"] = train_df["dealer_id"].map(cmap).fillna("unknown")
    test_df["dealer_cluster"]  = test_df["dealer_id"].map(cmap).fillna("unknown")

    return train_df, test_df


# ════════════════════════════════════════════════════════════════════════════
# model.py  — Fast CatBoost Training (NO LOG TRANSFORM)
# ════════════════════════════════════════════════════════════════════════════

def _prep_X(X: pd.DataFrame, cat_cols: list[str]) -> pd.DataFrame:
    X = X.copy()
    for c in cat_cols:
        if c in X.columns:
            X[c] = X[c].fillna("missing").astype(str)
    return X


def train_model(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    cat_cols: list[str],
    loss_function: str = "MAE",
    val_fraction: float = 0.20,
    sample_weights: np.ndarray = None,
) -> CatBoostRegressor:
    cat_indices = [i for i, c in enumerate(X_train.columns) if c in cat_cols]
    X_tr = _prep_X(X_train, cat_cols)

    # ── LOG TRANSFORM REMOVED: Training natively on car units for WAPE prioritization ──
    n_val = max(1, int(len(X_tr) * val_fraction))
    X_fit, y_fit = X_tr.iloc[:-n_val], y_train.iloc[:-n_val]
    X_val, y_val = X_tr.iloc[-n_val:], y_train.iloc[-n_val:]

    w_fit = sample_weights[:-n_val] if sample_weights is not None else None

    # Fast robust defaults
    model = CatBoostRegressor(
        iterations            = 800,
        learning_rate         = 0.04,
        depth                 = 6,
        l2_leaf_reg           = 3.0,
        loss_function         = loss_function,
        early_stopping_rounds = 50,
        random_seed           = 42,
        verbose               = 0,
        cat_features          = cat_indices,
    )

    model.fit(X_fit, y_fit, sample_weight=w_fit, eval_set=(X_val, y_val), use_best_model=True)
    return model


def predict(model: CatBoostRegressor, X_test: pd.DataFrame, cat_cols: list[str]) -> np.ndarray:
    X_te = _prep_X(X_test, cat_cols)
    raw_preds = model.predict(X_te)
    return np.maximum(0.0, raw_preds)


def compute_optimal_po(p50: np.ndarray, p85: np.ndarray, horizon: int, safety_multipliers: dict = SAFETY_MULTIPLIERS) -> np.ndarray:
    multiplier = safety_multipliers.get(horizon, 1.12)
    po = p50 * multiplier
    po = np.minimum(po, p85)
    return np.maximum(0, np.round(po)).astype(int)


# ════════════════════════════════════════════════════════════════════════════
# run.py  & Validation
# ════════════════════════════════════════════════════════════════════════════

def walk_forward_validate(df: pd.DataFrame, horizon: int, features: list[str], cat_feats: list[str], n_windows: int = VALIDATION_WINDOWS) -> dict:
    all_months = sorted(df["year_month"].unique())
    if len(all_months) < n_windows + horizon:
        n_windows = max(1, len(all_months) - horizon)

    test_months = all_months[-(n_windows + horizon):-horizon] or [all_months[-horizon-1]]
    records = []

    target_col = f"target_h{horizon}"
    df_cv = df.copy()
    df_cv[target_col] = df_cv.groupby(["dealer_id", "model", "variant"])["true_demand"].shift(-(horizon - 1))

    for test_month in test_months:
        train = df_cv[(df_cv["year_month"] < test_month) & df_cv[target_col].notna()].copy()
        test_features = df_cv[df_cv["year_month"] == test_month].copy()

        future_months = all_months
        test_idx = future_months.index(test_month)
        target_month = future_months[test_idx + horizon - 1]

        target = df_cv[df_cv["year_month"] == target_month][["dealer_id", "model", "variant", "true_demand"]].rename(columns={"true_demand": "actual"})

        train, test_features = assign_dealer_clusters(train, test_features)

        hist = train.groupby(["dealer_id", "model", "variant"])["year_month"].nunique()
        sufficient = set(hist[hist >= MIN_HISTORY_MONTHS].index)

        feats_avail = [f for f in features if f in test_features.columns]
        cat_avail   = [c for c in cat_feats if c in feats_avail]

        mask = test_features.set_index(["dealer_id", "model", "variant"]).index.isin(sufficient)
        test_sub = test_features[mask].copy()

        if len(test_sub) == 0: continue

        # ── Apply Outlier Weights strictly in validation to reflect real behavior ──
        if "supply_shock_flag" in train.columns:
            val_weights = np.where(train["supply_shock_flag"].fillna(0) == 1, 0.4, 1.0)
        else:
            val_weights = None

        model_p50 = train_model(train[feats_avail], train[target_col], cat_avail, "MAE", sample_weights=val_weights)
        preds = predict(model_p50, test_sub[feats_avail], cat_avail)

        eval_df = test_sub[["dealer_id", "model", "variant"]].copy()
        eval_df["predicted"] = preds
        eval_df = eval_df.merge(target, on=["dealer_id", "model", "variant"], how="inner")

        y_true = eval_df["actual"].to_numpy()
        y_pred = eval_df["predicted"].to_numpy()

        wape = np.sum(np.abs(y_pred - y_true)) / (np.sum(y_true) + 1e-9) * 100
        mae  = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))

        records.append({"test_month": test_month, "target_month": target_month, "n_dealers": len(eval_df), "wape": wape, "mae": mae, "rmse": rmse})

    wapes = [r["wape"] for r in records] if records else []
    return {
        "windows": records,
        "mean_wape": np.mean(wapes) if wapes else None,
        "std_wape":  np.std(wapes) if wapes else None,
    }


def _ets_fallback_forecast(train: pd.DataFrame, test_keys: set, horizon: int) -> dict:
    preds = {}
    for (did, mod, var), grp in train.groupby(["dealer_id", "model", "variant"]):
        if (did, mod, var) not in test_keys: continue
        s = grp["true_demand"].dropna()
        if len(s) < 12:
            preds[(did, mod, var)] = max(0.0, s.mean()) if len(s) > 0 else 0.0
            continue
        try:
            model = ExponentialSmoothing(s, trend='add', seasonal=None).fit(optimized=True)
            preds[(did, mod, var)] = max(0.0, float(model.forecast(horizon).iloc[-1]))
        except Exception:
            preds[(did, mod, var)] = max(0.0, s.tail(3).mean())
    return preds


def run_pipeline(
    dealer_fact_path   = "fact_dealer_monthly.csv",
    dealer_dim_path    = "dim_dealers.csv",
    model_dim_path     = "dim_models.csv",
    model_monthly_path = "fact_model_monthly.csv",
    external_path      = "fact_external_monthly.csv",
    production_path    = "fact_production_monthly.csv",
    run_validation     = True,
):
    print("Loading and merging source tables...")
    df = load_and_merge(dealer_fact_path, dealer_dim_path, model_dim_path, model_monthly_path, external_path, production_path)
    print("Reconstructing true retail demand...")
    df = reconstruct_true_demand(df)
    print("Engineering features...")
    df = build_features(df)
    df = df.dropna(subset=["demand_lag12"]).reset_index(drop=True)

    feats_avail = [f for f in FEATURES if f in df.columns]
    cat_avail   = [c for c in CAT_FEATURES if c in feats_avail]
    val_metrics = {}

    if run_validation:
        print("Running walk-forward validation (Fast Mode)...")
        for H in HORIZONS:
            print(f"  Validating H={H}M...", end=" ", flush=True)
            m = walk_forward_validate(df, H, feats_avail, cat_avail)
            val_metrics[H] = m
            if m["mean_wape"] is not None:
                print(f"WAPE = {m['mean_wape']:.1f}% ± {m['std_wape']:.1f}pp")
            else:
                print("insufficient data")

    all_months = sorted(df["year_month"].unique())
    test_month = all_months[-1]

    print(f"\nFinal forecast month: {test_month}")
    all_results = []

    for H in HORIZONS:
        print(f"\n── Horizon H={H}M ──────────────────────────────────────")

        target_col = f"target_h{H}"
        df[target_col] = df.groupby(["dealer_id", "model", "variant"])["true_demand"].shift(-(H - 1))

        train_df = df[(df["year_month"] < test_month) & df[target_col].notna()].copy()
        test_df  = df[df["year_month"] == test_month].copy()
        train_df, test_df = assign_dealer_clusters(train_df, test_df)

        hist_len   = train_df.groupby(["dealer_id", "model", "variant"])["year_month"].nunique()
        thin_keys  = set(hist_len[hist_len < MIN_HISTORY_MONTHS].index.tolist())

        sufficient_mask = ~test_df.set_index(["dealer_id", "model", "variant"]).index.isin(thin_keys)
        test_suf = test_df[sufficient_mask].copy()
        test_thin = test_df[~sufficient_mask].copy()
        result_rows = []

        if len(test_suf) > 0:
            print(f"  Training models for exactly {H} month(s) ahead...")

            # ── Calculating Outlier Weights ──
            if "supply_shock_flag" in train_df.columns:
                robust_weights = np.where(train_df["supply_shock_flag"].fillna(0) == 1, 0.4, 1.0)
            else:
                robust_weights = np.ones(len(train_df))

            # ── PRUNING: Scout model determines what to drop ──
            model_scout = train_model(train_df[feats_avail], train_df[target_col], cat_avail, "MAE", sample_weights=robust_weights)
            imp = model_scout.get_feature_importance(prettified=True)
            dead_features = imp.tail(int(len(imp) * 0.20))["Feature Id"].tolist()

            feats_pruned = [f for f in feats_avail if f not in dead_features]
            cat_pruned   = [c for c in cat_avail if c in feats_pruned]

            # ── Train FINAL models with weights ──
            model_p50 = train_model(train_df[feats_pruned], train_df[target_col], cat_pruned, "MAE", sample_weights=robust_weights)
            model_p85 = train_model(train_df[feats_pruned], train_df[target_col], cat_pruned, "Quantile:alpha=0.85", sample_weights=robust_weights)

            p50 = predict(model_p50, test_suf[feats_pruned], cat_pruned)
            p85 = predict(model_p85, test_suf[feats_pruned], cat_pruned)
            po  = compute_optimal_po(p50, p85, H)

            suf_out = test_suf[[
                "dealer_id", "model", "variant", "region", "zone", "city", "tier_city",
                "segment", "fuel_type", "net_bookings", "cancel_rate", "true_demand",
                "inv_lag1", "fill_rate_lag1", "coverage_days", "dealer_infl_roll3",
                "panic_order_lag1", "dealer_requested_qty",
            ]].copy()

            suf_out["predicted_demand"]  = np.round(p50).astype(int)
            suf_out["po_upper_bound"]    = np.round(p85).astype(int)
            suf_out["optimal_po"]        = po
            suf_out["horizon_months"]    = H
            suf_out["forecast_method"]   = "catboost"
            suf_out["forecast_year_month"] = test_month
            result_rows.append(suf_out)

        if len(test_thin) > 0:
            rb_preds = _ets_fallback_forecast(train_df, thin_keys, H)
            thin_out = test_thin[[
                "dealer_id", "model", "variant", "region", "zone", "city", "tier_city",
                "segment", "fuel_type", "net_bookings", "cancel_rate", "true_demand",
                "inv_lag1", "fill_rate_lag1", "coverage_days", "dealer_infl_roll3",
                "panic_order_lag1", "dealer_requested_qty",
            ]].copy()

            rb_vals = np.array([rb_preds.get((r.dealer_id, r.model, r.variant), 0.0) for r in test_thin.itertuples()])
            thin_out["predicted_demand"]  = np.round(rb_vals).astype(int)
            thin_out["po_upper_bound"]    = np.round(rb_vals * 1.20).astype(int)
            thin_out["optimal_po"]        = np.round(rb_vals * SAFETY_MULTIPLIERS[H]).astype(int)
            thin_out["horizon_months"]    = H
            thin_out["forecast_method"]   = "ets_fallback"
            thin_out["forecast_year_month"] = test_month
            result_rows.append(thin_out)

        if result_rows:
            horizon_result = pd.concat(result_rows, ignore_index=True)
            all_results.append(horizon_result)

            if H == 1 and len(suf_out) > 0 and "true_demand" in suf_out.columns:
                print("  Metrics (H=1 only, ground truth available):")
                y_true = suf_out["true_demand"].to_numpy()
                y_pred = suf_out["predicted_demand"].to_numpy()
                y_po   = suf_out["optimal_po"].to_numpy()
                wape = np.sum(np.abs(y_pred - y_true)) / (np.sum(y_true) + 1e-9) * 100
                fill_rate = (y_po >= y_true).mean() * 100
                print(f"  WAPE: {wape:.1f}% | Fill rate: {fill_rate:.1f}%")
            elif H > 1:
                print(f"  Predictions generated for Month +{H}.")

    results = pd.concat(all_results, ignore_index=True)

    target_dates = []
    for _, row in results.iterrows():
        origin_date = pd.to_datetime(str(row["forecast_year_month"]) + "-01")
        target_dates.append((origin_date + pd.DateOffset(months=row["horizon_months"])).strftime("%Y-%m"))

    results["target_forecast_month"] = target_dates
    results = results.rename(columns={"forecast_year_month": "forecast_origin_month"})

    output_cols = [
        "dealer_id", "region", "zone", "city", "tier_city", "model", "variant", "segment", "fuel_type",
        "horizon_months", "forecast_origin_month", "target_forecast_month", "forecast_method",
        "enquiries", "test_drives", "net_bookings", "cancel_rate", "repeat_orders",
        "testdrive_to_booking_ratio", "competitor_segment_launch_lag1",
        "true_demand", "predicted_demand", "po_upper_bound", "optimal_po",
        "dealer_requested_qty", "dealer_infl_roll3", "panic_order_lag1", "inv_lag1", "fill_rate_lag1", "coverage_days",
    ]
    output_cols = [c for c in output_cols if c in results.columns]
    results = results.sort_values(["dealer_id", "model", "variant", "horizon_months"])

    results[output_cols].to_csv("dealer_demand_forecast.csv", index=False)

    # Generate aggregate plans
    plan = results.groupby(["model", "variant", "target_forecast_month", "horizon_months"]).agg(
        total_predicted_demand=("predicted_demand", "sum"), total_optimal_po=("optimal_po", "sum")
    ).reset_index()
    plan.to_csv("monthly_production_plan.csv", index=False)

    regional = results.groupby(["region", "zone", "model", "variant", "target_forecast_month", "horizon_months"]).agg(
        regional_predicted=("predicted_demand", "sum"), regional_po=("optimal_po", "sum")
    ).reset_index()
    regional.to_csv("regional_allocation_plan.csv", index=False)

    print("\n── Outputs saved ───────────────────────────────────────────────")
    return results, val_metrics

if __name__ == "__main__":
    run_pipeline()

Loading and merging source tables...
Reconstructing true retail demand...
Engineering features...
Running walk-forward validation (Fast Mode)...
  Validating H=1M... WAPE = 30.3% ± 1.5pp
  Validating H=3M... WAPE = 31.9% ± 1.2pp
  Validating H=6M... WAPE = 32.0% ± 1.1pp

Final forecast month: 2023-12

── Horizon H=1M ──────────────────────────────────────
  Training models for exactly 1 month(s) ahead...
  Metrics (H=1 only, ground truth available):
  WAPE: 32.0% | Fill rate: 71.7%

── Horizon H=3M ──────────────────────────────────────
  Training models for exactly 3 month(s) ahead...
  Predictions generated for Month +3.

── Horizon H=6M ──────────────────────────────────────
  Training models for exactly 6 month(s) ahead...
  Predictions generated for Month +6.

── Outputs saved ───────────────────────────────────────────────


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.1 MB/s eta 0:00:00


In [ ]:
!pip install lightgbm

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
import lightgbm as lgb
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ════════════════════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════

CAT_FEATURES = [
    "dealer_id", "region", "tier_city", "zone",
    "model", "variant", "segment", "fuel_type", "transmission",
    "allocation_policy", "credit_rating",
    "dealer_cluster",
]

FEATURES = [
    "dealer_id", "region", "tier_city", "zone",
    "model", "variant", "segment", "fuel_type", "transmission",
    "allocation_policy", "credit_rating", "dealer_cluster",

    "dealership_age_years", "num_sales_staff", "showroom_area_sqft",

    "demand_lag1", "demand_lag3", "demand_lag6", "demand_lag12",
    "demand_roll3", "demand_roll6",
    "demand_trend3", "yoy_demand_growth",

    "demand_roll12", "demand_roll12_std", "yoy_roll_ratio",

    "enquiries_lag1", "test_drives_lag1",
    "enquiry_to_testdrive_ratio", "testdrive_to_booking_ratio",
    "repeat_order_pct",

    "bookings_lag1", "booking_roll3",
    "cancel_rate_roll3",
    "conversion_rate_roll6",
    "booking_implied_demand",

    "inv_lag1", "fill_rate_lag1", "coverage_days", "supply_shock_flag",

    "dealer_infl_roll3", "panic_order_lag1",

    "current_price_inr", "discount_offered_inr",
    "price_change_lag1", "campaign_active",
    "months_since_launch", "new_variant_flag",
    "competitor_segment_launch_lag1",

    "fuel_price_inr_per_litre",
    "festive_month_flag", "holiday_count",
    "regional_gdp_growth_pct", "rainfall_mm",

    "sin_month_k1", "cos_month_k1",
    "sin_month_k2", "cos_month_k2",
    "sin_month_k3", "cos_month_k3",
    "year",
]

HORIZONS = [1, 3, 6]

STOCKOUT_COST_RATIO = 0.30
HOLDING_COST_RATIO  = 0.05

SAFETY_MULTIPLIERS = {1: 1.18, 3: 1.12, 6: 1.08}
MIN_HISTORY_MONTHS = 12
VALIDATION_WINDOWS = 6


# ════════════════════════════════════════════════════════════════════════════
# 2. DATA LOADING & MERGING
# ════════════════════════════════════════════════════════════════════════════

def load_and_merge(
    dealer_fact_path   = "fact_dealer_monthly.csv",
    dealer_dim_path    = "dim_dealers.csv",
    model_dim_path     = "dim_models.csv",
    model_monthly_path = "fact_model_monthly.csv",
    external_path      = "fact_external_monthly.csv",
    production_path    = "fact_production_monthly.csv",
) -> pd.DataFrame:
    dealer_fact = pd.read_csv(dealer_fact_path)
    dealer_dim  = pd.read_csv(dealer_dim_path)
    model_dim   = pd.read_csv(model_dim_path)
    model_mnth  = pd.read_csv(model_monthly_path)
    external    = pd.read_csv(external_path)
    production  = pd.read_csv(production_path)

    dealer_dim = dealer_dim.drop(columns=["allocation_policy"], errors="ignore")

    df = dealer_fact.merge(
        dealer_dim[[
            "dealer_id", "region", "city", "tier_city", "zone",
            "dealership_age_years", "showroom_area_sqft",
            "service_center", "num_sales_staff", "credit_rating",
        ]], on="dealer_id", how="left"
    ).merge(
        model_mnth.drop(columns=["year", "month"], errors="ignore"), on=["model", "variant", "year_month"], how="left"
    ).merge(
        external.drop(columns=["year", "month"], errors="ignore"), on=["region", "year_month"], how="left"
    ).merge(
        production[[
            "model", "variant", "year_month",
            "monthly_capacity", "planned_production",
            "actual_production", "supply_shock_flag",
        ]], on=["model", "variant", "year_month"], how="left"
    ).merge(
        model_dim[["model", "variant", "segment", "fuel_type", "transmission"]], on=["model", "variant"], how="left"
    )

    return df.sort_values(["dealer_id", "model", "variant", "year", "month"]).reset_index(drop=True)


# ════════════════════════════════════════════════════════════════════════════
# 3. FEATURE ENGINEERING
# ════════════════════════════════════════════════════════════════════════════

def reconstruct_true_demand(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(["dealer_id", "model", "variant"])

    df["true_demand"] = (df["retail_sales"] + df["opening_inventory"] - df["closing_inventory"]).clip(lower=0)
    df["demand_unbiased"] = np.where(df["stockout_flag"] == 1, np.nan, df["true_demand"])
    df["true_demand"] = g["demand_unbiased"].transform(lambda s: s.fillna(s.rolling(3, min_periods=1).mean())).fillna(df["true_demand"])

    df["net_bookings"] = (df["bookings"] - df["cancellations"]).clip(lower=0)
    df["cancel_rate"]  = df["cancellations"] / (df["bookings"] + 1)
    df["dealer_inflation"] = df["dealer_requested_qty"] - df["true_demand"]
    df["dealer_infl_roll3"] = g["dealer_inflation"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    df["panic_order_lag1"] = g["panic_order_flag"].shift(1).fillna(0)

    return df

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(["dealer_id", "model", "variant"])

    for lag in [1, 3, 6, 12]:
        df[f"demand_lag{lag}"] = g["true_demand"].shift(lag)

    for window in [3, 6]:
        df[f"demand_roll{window}"] = g["true_demand"].shift(1).transform(lambda s: s.rolling(window, min_periods=1).mean())

    df["demand_roll12"] = g["true_demand"].shift(1).transform(lambda s: s.rolling(12, min_periods=6).mean())
    df["demand_roll12_std"] = g["true_demand"].shift(1).transform(lambda s: s.rolling(12, min_periods=6).std()).fillna(0)
    df["yoy_roll_ratio"] = df["demand_roll3"] / (df["demand_roll12"] + 1)

    df["demand_trend3"]     = (df["demand_lag1"] - df["demand_lag3"]) / 2.0
    df["yoy_demand_growth"] = (df["demand_lag1"] - df["demand_lag12"]) / (df["demand_lag12"] + 1)

    df["bookings_lag1"] = g["net_bookings"].shift(1)
    df["booking_roll3"] = g["net_bookings"].shift(1).transform(lambda s: s.rolling(3, min_periods=1).mean())
    df["cancel_rate_roll3"] = g["cancel_rate"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

    df["conversion_rate_roll6"] = (
        g["retail_sales"].shift(1).transform(lambda s: s.rolling(6, min_periods=1).sum()) /
        (g["bookings"].shift(1).transform(lambda s: s.rolling(6, min_periods=1).sum()) + 1)
    )
    df["booking_implied_demand"] = df["bookings_lag1"] * df["conversion_rate_roll6"]

    if "enquiries" in df.columns and "test_drives" in df.columns:
        df["enquiries_lag1"] = g["enquiries"].shift(1)
        df["test_drives_lag1"] = g["test_drives"].shift(1)
        df["enquiry_to_testdrive_ratio"] = g["test_drives"].shift(1) / (g["enquiries"].shift(1) + 1)
        df["testdrive_to_booking_ratio"] = g["bookings"].shift(1) / (g["test_drives"].shift(1) + 1)
        df["repeat_order_pct"] = g["repeat_orders"].shift(1) / (g["retail_sales"].shift(1) + 1)

    df["inv_lag1"]       = g["closing_inventory"].shift(1)
    df["fill_rate_lag1"] = g["allocated_qty"].shift(1) / (g["dealer_requested_qty"].shift(1) + 1)
    df["coverage_days"]  = df["inv_lag1"] / (df["demand_roll3"] / 30.0 + 0.001)

    df["price_change_lag1"] = g["price_change_pct"].shift(1)
    if "competitor_segment_launch" in df.columns:
        df["competitor_segment_launch_lag1"] = g["competitor_segment_launch"].shift(1)

    for k in [1, 2, 3]:
        df[f"sin_month_k{k}"] = np.sin(2 * np.pi * k * df["month"] / 12)
        df[f"cos_month_k{k}"] = np.cos(2 * np.pi * k * df["month"] / 12)

    return df

def assign_dealer_clusters(train_df: pd.DataFrame, test_df: pd.DataFrame, n_clusters=6):
    cluster_feats = ["demand_roll12", "cancel_rate_roll3", "dealer_infl_roll3", "fill_rate_lag1"]
    avail = [f for f in cluster_feats if f in train_df.columns]

    if not avail:
        train_df["dealer_cluster"] = "unknown"
        test_df["dealer_cluster"]  = "unknown"
        return train_df, test_df

    profiles = train_df.groupby("dealer_id")[avail].mean().fillna(0).reset_index()
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    profiles["dealer_cluster"] = km.fit_predict(profiles[avail]).astype(str)

    cmap = profiles.set_index("dealer_id")["dealer_cluster"].to_dict()
    train_df["dealer_cluster"] = train_df["dealer_id"].map(cmap).fillna("unknown")
    test_df["dealer_cluster"]  = test_df["dealer_id"].map(cmap).fillna("unknown")

    return train_df, test_df


# ════════════════════════════════════════════════════════════════════════════
# 4. MODELING (HURDLE + ENSEMBLE + HIERARCHICAL)
# ════════════════════════════════════════════════════════════════════════════

def _prep_X(X: pd.DataFrame, cat_cols: list[str], for_lgbm: bool = False) -> pd.DataFrame:
    X = X.copy()
    for c in cat_cols:
        if c in X.columns:
            if for_lgbm:
                X[c] = X[c].fillna("missing").astype("category")
            else:
                X[c] = X[c].fillna("missing").astype(str)
    return X

def train_hurdle_ensemble(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    cat_cols: list[str],
    val_fraction: float = 0.20,
    sample_weights: np.ndarray = None,
):
    n_val = max(1, int(len(X_train) * val_fraction))
    X_fit, y_fit = X_train.iloc[:-n_val].copy(), y_train.iloc[:-n_val].copy()
    X_val, y_val = X_train.iloc[-n_val:].copy(), y_train.iloc[-n_val:].copy()
    w_fit = sample_weights[:-n_val] if sample_weights is not None else None

    y_fit_bin = (y_fit > 0).astype(int)
    y_val_bin = (y_val > 0).astype(int)

    # ── Stage 1: Classifier ──
    X_fit_cb = _prep_X(X_fit, cat_cols)
    X_val_cb = _prep_X(X_val, cat_cols)
    cat_indices = [i for i, c in enumerate(X_fit_cb.columns) if c in cat_cols]

    classifier = CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=5,
        loss_function='Logloss', early_stopping_rounds=30,
        random_seed=42, verbose=0, cat_features=cat_indices
    )
    classifier.fit(X_fit_cb, y_fit_bin, sample_weight=w_fit, eval_set=(X_val_cb, y_val_bin), use_best_model=True)

    # ── Stage 2: Regressors (Demand > 0 ONLY) ──
    mask_fit_nz = y_fit > 0
    mask_val_nz = y_val > 0

    X_fit_nz, y_fit_nz = X_fit[mask_fit_nz], y_fit[mask_fit_nz]
    X_val_nz, y_val_nz = X_val[mask_val_nz], y_val[mask_val_nz]
    w_fit_nz = w_fit[mask_fit_nz] if w_fit is not None else None

    cb_regressor, lgb_regressor = None, None

    if len(X_fit_nz) > 20:
        # CatBoost Tweedie
        cb_regressor = CatBoostRegressor(
            iterations=800, learning_rate=0.03, depth=6, l2_leaf_reg=4.0,
            loss_function='Tweedie:variance_power=1.5',
            early_stopping_rounds=50, random_seed=42, verbose=0, cat_features=cat_indices
        )
        cb_regressor.fit(
            _prep_X(X_fit_nz, cat_cols), y_fit_nz,
            sample_weight=w_fit_nz,
            eval_set=(_prep_X(X_val_nz, cat_cols), y_val_nz),
            use_best_model=True
        )

        # LightGBM Tweedie
        X_fit_lgb = _prep_X(X_fit_nz, cat_cols, for_lgbm=True)
        X_val_lgb = _prep_X(X_val_nz, cat_cols, for_lgbm=True)

        lgb_train = lgb.Dataset(X_fit_lgb, label=y_fit_nz, weight=w_fit_nz)
        lgb_val = lgb.Dataset(X_val_lgb, label=y_val_nz, reference=lgb_train)

        lgb_params = {
            'objective': 'tweedie',
            'tweedie_variance_power': 1.5,
            'metric': 'mae',
            'learning_rate': 0.03,
            'max_depth': 6,
            'num_leaves': 31,
            'verbose': -1,
            'seed': 42
        }

        lgb_regressor = lgb.train(
            lgb_params, lgb_train, num_boost_round=800,
            valid_sets=[lgb_val],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

    return {
        "classifier": classifier,
        "cb_regressor": cb_regressor,
        "lgb_regressor": lgb_regressor
    }

def predict_hurdle_ensemble(models: dict, X_test: pd.DataFrame, cat_cols: list[str], lgbm_weight: float = 0.4) -> np.ndarray:
    classifier = models["classifier"]
    cb_reg = models["cb_regressor"]
    lgb_reg = models["lgb_regressor"]

    X_cb = _prep_X(X_test, cat_cols)
    prob_positive = classifier.predict_proba(X_cb)[:, 1]

    if cb_reg is not None and lgb_reg is not None:
        cb_preds = np.maximum(0.0, cb_reg.predict(X_cb))
        X_lgb = _prep_X(X_test, cat_cols, for_lgbm=True)
        lgb_preds = np.maximum(0.0, lgb_reg.predict(X_lgb))
        ensemble_volume = (lgbm_weight * lgb_preds) + ((1 - lgbm_weight) * cb_preds)
    else:
        ensemble_volume = np.zeros(len(X_test))

    return np.maximum(0.0, prob_positive * ensemble_volume)

# Legacy standard model (Kept strictly for Scout pruning and P85 Quantile Safety Stock)
def train_model(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    cat_cols: list[str],
    loss_function: str = "MAE",
    val_fraction: float = 0.20,
    sample_weights: np.ndarray = None,
) -> CatBoostRegressor:
    cat_indices = [i for i, c in enumerate(X_train.columns) if c in cat_cols]
    X_tr = _prep_X(X_train, cat_cols)

    n_val = max(1, int(len(X_tr) * val_fraction))
    X_fit, y_fit = X_tr.iloc[:-n_val], y_train.iloc[:-n_val]
    X_val, y_val = X_tr.iloc[-n_val:], y_train.iloc[-n_val:]
    w_fit = sample_weights[:-n_val] if sample_weights is not None else None

    model = CatBoostRegressor(
        iterations=800, learning_rate=0.04, depth=6, l2_leaf_reg=3.0,
        loss_function=loss_function, early_stopping_rounds=50,
        random_seed=42, verbose=0, cat_features=cat_indices,
    )
    model.fit(X_fit, y_fit, sample_weight=w_fit, eval_set=(X_val, y_val), use_best_model=True)
    return model

def predict(model: CatBoostRegressor, X_test: pd.DataFrame, cat_cols: list[str]) -> np.ndarray:
    X_te = _prep_X(X_test, cat_cols)
    return np.maximum(0.0, model.predict(X_te))

def compute_optimal_po(p50: np.ndarray, p85: np.ndarray, horizon: int, safety_multipliers: dict = SAFETY_MULTIPLIERS) -> np.ndarray:
    multiplier = safety_multipliers.get(horizon, 1.12)
    po = p50 * multiplier
    po = np.minimum(po, p85)
    return np.maximum(0, np.round(po)).astype(int)

def reconcile_hierarchical_forecasts(df: pd.DataFrame, preds_df: pd.DataFrame) -> pd.DataFrame:
    hist = df[df["true_demand"] > 0].copy()
    dealer_vol = hist.groupby(["zone", "model", "variant", "dealer_id"])["true_demand"].sum().reset_index(name="dealer_total")
    zone_vol = hist.groupby(["zone", "model", "variant"])["true_demand"].sum().reset_index(name="zone_total")

    shares = dealer_vol.merge(zone_vol, on=["zone", "model", "variant"])
    shares["historical_share"] = shares["dealer_total"] / (shares["zone_total"] + 1e-9)

    merged = preds_df.merge(shares[["zone", "model", "variant", "dealer_id", "historical_share"]],
                            on=["zone", "model", "variant", "dealer_id"], how="left")
    merged["historical_share"] = merged["historical_share"].fillna(0)

    zone_preds = merged.groupby(["zone", "model", "variant"])["predicted_demand"].sum().reset_index(name="zone_pred_total")
    merged = merged.merge(zone_preds, on=["zone", "model", "variant"], how="left")

    merged["top_down_pred"] = merged["zone_pred_total"] * merged["historical_share"]
    merged["reconciled_demand"] = (merged["predicted_demand"] * 0.5) + (merged["top_down_pred"] * 0.5)
    merged["predicted_demand"] = np.round(merged["reconciled_demand"]).astype(int)

    return merged.drop(columns=["historical_share", "zone_pred_total", "top_down_pred", "reconciled_demand"])


# ════════════════════════════════════════════════════════════════════════════
# 5. PIPELINE EXECUTION
# ════════════════════════════════════════════════════════════════════════════

def walk_forward_validate(df: pd.DataFrame, horizon: int, features: list[str], cat_feats: list[str], n_windows: int = VALIDATION_WINDOWS) -> dict:
    all_months = sorted(df["year_month"].unique())
    if len(all_months) < n_windows + horizon:
        n_windows = max(1, len(all_months) - horizon)

    test_months = all_months[-(n_windows + horizon):-horizon] or [all_months[-horizon-1]]
    records = []

    target_col = f"target_h{horizon}"
    df_cv = df.copy()
    df_cv[target_col] = df_cv.groupby(["dealer_id", "model", "variant"])["true_demand"].shift(-(horizon - 1))

    for test_month in test_months:
        train = df_cv[(df_cv["year_month"] < test_month) & df_cv[target_col].notna()].copy()
        test_features = df_cv[df_cv["year_month"] == test_month].copy()

        future_months = all_months
        test_idx = future_months.index(test_month)
        target_month = future_months[test_idx + horizon - 1]

        target = df_cv[df_cv["year_month"] == target_month][["dealer_id", "model", "variant", "true_demand"]].rename(columns={"true_demand": "actual"})

        train, test_features = assign_dealer_clusters(train, test_features)

        hist = train.groupby(["dealer_id", "model", "variant"])["year_month"].nunique()
        sufficient = set(hist[hist >= MIN_HISTORY_MONTHS].index)

        feats_avail = [f for f in features if f in test_features.columns]
        cat_avail   = [c for c in cat_feats if c in feats_avail]

        mask = test_features.set_index(["dealer_id", "model", "variant"]).index.isin(sufficient)
        test_sub = test_features[mask].copy()

        if len(test_sub) == 0: continue

        val_weights = np.where(train["supply_shock_flag"].fillna(0) == 1, 0.4, 1.0) if "supply_shock_flag" in train.columns else None

        # Validate with the new Hurdle Ensemble architecture
        models_p50 = train_hurdle_ensemble(train[feats_avail], train[target_col], cat_avail, sample_weights=val_weights)
        preds = predict_hurdle_ensemble(models_p50, test_sub[feats_avail], cat_avail)

        eval_df = test_sub[["dealer_id", "model", "variant"]].copy()
        eval_df["predicted"] = preds
        eval_df = eval_df.merge(target, on=["dealer_id", "model", "variant"], how="inner")

        y_true = eval_df["actual"].to_numpy()
        y_pred = eval_df["predicted"].to_numpy()

        wape = np.sum(np.abs(y_pred - y_true)) / (np.sum(y_true) + 1e-9) * 100
        mae  = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))

        records.append({"test_month": test_month, "target_month": target_month, "n_dealers": len(eval_df), "wape": wape, "mae": mae, "rmse": rmse})

    wapes = [r["wape"] for r in records] if records else []
    return {
        "windows": records,
        "mean_wape": np.mean(wapes) if wapes else None,
        "std_wape":  np.std(wapes) if wapes else None,
    }

def _ets_fallback_forecast(train: pd.DataFrame, test_keys: set, horizon: int) -> dict:
    preds = {}
    for (did, mod, var), grp in train.groupby(["dealer_id", "model", "variant"]):
        if (did, mod, var) not in test_keys: continue
        s = grp["true_demand"].dropna()
        if len(s) < 12:
            preds[(did, mod, var)] = max(0.0, s.mean()) if len(s) > 0 else 0.0
            continue
        try:
            model = ExponentialSmoothing(s, trend='add', seasonal=None).fit(optimized=True)
            preds[(did, mod, var)] = max(0.0, float(model.forecast(horizon).iloc[-1]))
        except Exception:
            preds[(did, mod, var)] = max(0.0, s.tail(3).mean())
    return preds

def run_pipeline(
    dealer_fact_path   = "fact_dealer_monthly.csv",
    dealer_dim_path    = "dim_dealers.csv",
    model_dim_path     = "dim_models.csv",
    model_monthly_path = "fact_model_monthly.csv",
    external_path      = "fact_external_monthly.csv",
    production_path    = "fact_production_monthly.csv",
    run_validation     = True,
):
    print("Loading and merging source tables...")
    df = load_and_merge(dealer_fact_path, dealer_dim_path, model_dim_path, model_monthly_path, external_path, production_path)
    print("Reconstructing true retail demand...")
    df = reconstruct_true_demand(df)
    print("Engineering features...")
    df = build_features(df)
    df = df.dropna(subset=["demand_lag12"]).reset_index(drop=True)

    feats_avail = [f for f in FEATURES if f in df.columns]
    cat_avail   = [c for c in CAT_FEATURES if c in feats_avail]
    val_metrics = {}

    if run_validation:
        print("Running walk-forward validation (Hurdle + Ensemble Mode)...")
        for H in HORIZONS:
            print(f"  Validating H={H}M...", end=" ", flush=True)
            m = walk_forward_validate(df, H, feats_avail, cat_avail)
            val_metrics[H] = m
            if m["mean_wape"] is not None:
                print(f"WAPE = {m['mean_wape']:.1f}% ± {m['std_wape']:.1f}pp")
            else:
                print("insufficient data")

    all_months = sorted(df["year_month"].unique())
    test_month = all_months[-1]

    print(f"\nFinal forecast month: {test_month}")
    all_results = []

    for H in HORIZONS:
        print(f"\n── Horizon H={H}M ──────────────────────────────────────")

        target_col = f"target_h{H}"
        df[target_col] = df.groupby(["dealer_id", "model", "variant"])["true_demand"].shift(-(H - 1))

        train_df = df[(df["year_month"] < test_month) & df[target_col].notna()].copy()
        test_df  = df[df["year_month"] == test_month].copy()
        train_df, test_df = assign_dealer_clusters(train_df, test_df)

        hist_len   = train_df.groupby(["dealer_id", "model", "variant"])["year_month"].nunique()
        thin_keys  = set(hist_len[hist_len < MIN_HISTORY_MONTHS].index.tolist())

        sufficient_mask = ~test_df.set_index(["dealer_id", "model", "variant"]).index.isin(thin_keys)
        test_suf = test_df[sufficient_mask].copy()
        test_thin = test_df[~sufficient_mask].copy()
        result_rows = []

        if len(test_suf) > 0:
            print(f"  Training Hurdle models for exactly {H} month(s) ahead...")

            robust_weights = np.where(train_df["supply_shock_flag"].fillna(0) == 1, 0.4, 1.0) if "supply_shock_flag" in train_df.columns else np.ones(len(train_df))

            # ── PRUNING: Standard Scout model determines what to drop ──
            model_scout = train_model(train_df[feats_avail], train_df[target_col], cat_avail, "MAE", sample_weights=robust_weights)
            imp = model_scout.get_feature_importance(prettified=True)
            dead_features = imp.tail(int(len(imp) * 0.20))["Feature Id"].tolist()

            feats_pruned = [f for f in feats_avail if f not in dead_features]
            cat_pruned   = [c for c in cat_avail if c in feats_pruned]

            # ── 1. Train the Final Hurdle + Ensemble P50 Model ──
            models_p50 = train_hurdle_ensemble(train_df[feats_pruned], train_df[target_col], cat_pruned, sample_weights=robust_weights)
            p50 = predict_hurdle_ensemble(models_p50, test_suf[feats_pruned], cat_pruned)

            # ── 2. Train Quantile P85 for Safety Stock (Standard CatBoost) ──
            model_p85 = train_model(train_df[feats_pruned], train_df[target_col], cat_pruned, "Quantile:alpha=0.85", sample_weights=robust_weights)
            p85 = predict(model_p85, test_suf[feats_pruned], cat_pruned)

            po = compute_optimal_po(p50, p85, H)

            suf_out = test_suf[[
                "dealer_id", "model", "variant", "region", "zone", "city", "tier_city",
                "segment", "fuel_type", "net_bookings", "cancel_rate", "true_demand",
                "inv_lag1", "fill_rate_lag1", "coverage_days", "dealer_infl_roll3",
                "panic_order_lag1", "dealer_requested_qty",
            ]].copy()

            suf_out["predicted_demand"]  = np.round(p50).astype(int)
            suf_out["po_upper_bound"]    = np.round(p85).astype(int)
            suf_out["optimal_po"]        = po
            suf_out["horizon_months"]    = H
            suf_out["forecast_method"]   = "hurdle_ensemble_reconciled"
            suf_out["forecast_year_month"] = test_month

            # ── 3. Apply Hierarchical Reconciliation ──
            suf_out = reconcile_hierarchical_forecasts(df, suf_out)
            result_rows.append(suf_out)

        if len(test_thin) > 0:
            rb_preds = _ets_fallback_forecast(train_df, thin_keys, H)
            thin_out = test_thin[[
                "dealer_id", "model", "variant", "region", "zone", "city", "tier_city",
                "segment", "fuel_type", "net_bookings", "cancel_rate", "true_demand",
                "inv_lag1", "fill_rate_lag1", "coverage_days", "dealer_infl_roll3",
                "panic_order_lag1", "dealer_requested_qty",
            ]].copy()

            rb_vals = np.array([rb_preds.get((r.dealer_id, r.model, r.variant), 0.0) for r in test_thin.itertuples()])
            thin_out["predicted_demand"]  = np.round(rb_vals).astype(int)
            thin_out["po_upper_bound"]    = np.round(rb_vals * 1.20).astype(int)
            thin_out["optimal_po"]        = np.round(rb_vals * SAFETY_MULTIPLIERS[H]).astype(int)
            thin_out["horizon_months"]    = H
            thin_out["forecast_method"]   = "ets_fallback"
            thin_out["forecast_year_month"] = test_month
            result_rows.append(thin_out)

        if result_rows:
            horizon_result = pd.concat(result_rows, ignore_index=True)
            all_results.append(horizon_result)

            if H == 1 and len(suf_out) > 0 and "true_demand" in suf_out.columns:
                print("  Metrics (H=1 only, ground truth available):")
                y_true = suf_out["true_demand"].to_numpy()
                y_pred = suf_out["predicted_demand"].to_numpy()
                y_po   = suf_out["optimal_po"].to_numpy()
                wape = np.sum(np.abs(y_pred - y_true)) / (np.sum(y_true) + 1e-9) * 100
                fill_rate = (y_po >= y_true).mean() * 100
                print(f"  WAPE: {wape:.1f}% | Fill rate: {fill_rate:.1f}%")
            elif H > 1:
                print(f"  Predictions generated for Month +{H}.")

    results = pd.concat(all_results, ignore_index=True)

    target_dates = []
    for _, row in results.iterrows():
        origin_date = pd.to_datetime(str(row["forecast_year_month"]) + "-01")
        target_dates.append((origin_date + pd.DateOffset(months=row["horizon_months"])).strftime("%Y-%m"))

    results["target_forecast_month"] = target_dates
    results = results.rename(columns={"forecast_year_month": "forecast_origin_month"})

    output_cols = [
        "dealer_id", "region", "zone", "city", "tier_city", "model", "variant", "segment", "fuel_type",
        "horizon_months", "forecast_origin_month", "target_forecast_month", "forecast_method",
        "enquiries", "test_drives", "net_bookings", "cancel_rate", "repeat_orders",
        "testdrive_to_booking_ratio", "competitor_segment_launch_lag1",
        "true_demand", "predicted_demand", "po_upper_bound", "optimal_po",
        "dealer_requested_qty", "dealer_infl_roll3", "panic_order_lag1", "inv_lag1", "fill_rate_lag1", "coverage_days",
    ]
    output_cols = [c for c in output_cols if c in results.columns]
    results = results.sort_values(["dealer_id", "model", "variant", "horizon_months"])

    results[output_cols].to_csv("dealer_demand_forecast.csv", index=False)

    plan = results.groupby(["model", "variant", "target_forecast_month", "horizon_months"]).agg(
        total_predicted_demand=("predicted_demand", "sum"), total_optimal_po=("optimal_po", "sum")
    ).reset_index()
    plan.to_csv("monthly_production_plan.csv", index=False)

    regional = results.groupby(["region", "zone", "model", "variant", "target_forecast_month", "horizon_months"]).agg(
        regional_predicted=("predicted_demand", "sum"), regional_po=("optimal_po", "sum")
    ).reset_index()
    regional.to_csv("regional_allocation_plan.csv", index=False)

    print("\n── Outputs saved ───────────────────────────────────────────────")
    return results, val_metrics

if __name__ == "__main__":
    run_pipeline()

Loading and merging source tables...
Reconstructing true retail demand...
Engineering features...
Running walk-forward validation (Hurdle + Ensemble Mode)...
  Validating H=1M... WAPE = 30.7% ± 1.2pp
  Validating H=3M... WAPE = 33.8% ± 2.4pp
  Validating H=6M... WAPE = 34.1% ± 1.8pp

Final forecast month: 2023-12

── Horizon H=1M ──────────────────────────────────────
  Training Hurdle models for exactly 1 month(s) ahead...
  Metrics (H=1 only, ground truth available):
  WAPE: 32.6% | Fill rate: 76.6%

── Horizon H=3M ──────────────────────────────────────
  Training Hurdle models for exactly 3 month(s) ahead...


KeyboardInterrupt: 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
import lightgbm as lgb
from sklearn.cluster import KMeans
from datetime import datetime
from dateutil.relativedelta import relativedelta

# ════════════════════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════

CAT_FEATURES = [
    "dealer_id", "region", "tier_city", "zone",
    "model", "variant", "segment", "fuel_type", "transmission",
    "credit_rating", "dealer_cluster",
]

FEATURES = [
    "dealer_id", "region", "tier_city", "zone",
    "model", "variant", "segment", "fuel_type", "transmission",
    "credit_rating", "dealer_cluster",
    "dealership_age_years", "num_sales_staff", "showroom_area_sqft",

    "demand_lag1", "demand_lag3", "demand_lag6", "demand_lag12",
    "demand_roll3", "demand_roll6", "demand_trend3", "yoy_demand_growth",
    "demand_roll12", "demand_roll12_std", "yoy_roll_ratio",

    "enquiries_lag1", "test_drives_lag1",
    "enquiry_to_testdrive_ratio", "testdrive_to_booking_ratio",

    "bookings_lag1", "booking_roll3", "cancel_rate_roll3",
    "conversion_rate_roll6", "booking_implied_demand",

    "inv_lag1", "fill_rate_lag1", "coverage_days", "supply_shock_flag",
    "dealer_infl_roll3", "panic_order_lag1",

    "finance_penetration_rate", "months_since_launch", "months_since_launch_sq",
    "price_change_lag1", "competitor_segment_launch_lag1",
    "festive_month_flag", "post_festive_trough",

    "sin_month_k1", "cos_month_k1",
    "sin_month_k2", "cos_month_k2",
]

HORIZONS = [1, 3, 6, 12]
SAFETY_MULTIPLIERS = {1: 1.18, 3: 1.12, 6: 1.08, 12: 1.05}
MIN_HISTORY_MONTHS = 12
VALIDATION_WINDOWS = 1

def offset_month(ym_str: str, delta: int) -> str:
    """Helper to enforce strict time-gaps for validation."""
    dt = datetime.strptime(ym_str, "%Y-%m") + relativedelta(months=delta)
    return dt.strftime("%Y-%m")


# ════════════════════════════════════════════════════════════════════════════
# 2. DATA LOADING & MERGING
# ════════════════════════════════════════════════════════════════════════════

def load_and_merge(
    dealer_fact_path   = "fact_dealer_monthly.csv",
    dealer_dim_path    = "dim_dealers.csv",
    model_dim_path     = "dim_models.csv",
    model_monthly_path = "fact_model_monthly.csv",
    external_path      = "fact_external_monthly.csv",
    production_path    = "fact_production_monthly.csv",
) -> pd.DataFrame:
    try:
        dealer_fact = pd.read_csv(dealer_fact_path)
        dealer_dim  = pd.read_csv(dealer_dim_path)
        model_dim   = pd.read_csv(model_dim_path)
        model_mnth  = pd.read_csv(model_monthly_path)
        external    = pd.read_csv(external_path)
        production  = pd.read_csv(production_path)
    except FileNotFoundError:
        print("Warning: Missing CSVs. Ensure your paths are correct. Returning empty DataFrame for compilation check...")
        return pd.DataFrame()

    dealer_dim = dealer_dim.drop(columns=["allocation_policy"], errors="ignore")
    df = dealer_fact.merge(
        dealer_dim[["dealer_id", "region", "city", "tier_city", "zone", "dealership_age_years", "showroom_area_sqft", "num_sales_staff", "credit_rating"]],
        on="dealer_id", how="left"
    ).merge(
        model_mnth.drop(columns=["year", "month"], errors="ignore"), on=["model", "variant", "year_month"], how="left"
    ).merge(
        external.drop(columns=["year", "month"], errors="ignore"), on=["region", "year_month"], how="left"
    ).merge(
        production[["model", "variant", "year_month", "supply_shock_flag"]], on=["model", "variant", "year_month"], how="left"
    ).merge(
        model_dim[["model", "variant", "segment", "fuel_type", "transmission"]], on=["model", "variant"], how="left"
    )

    return df.sort_values(["dealer_id", "model", "variant", "year", "month"]).reset_index(drop=True)


# ════════════════════════════════════════════════════════════════════════════
# 3. FEATURE ENGINEERING
# ════════════════════════════════════════════════════════════════════════════

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty: return df
    g = df.groupby(["dealer_id", "model", "variant"])

    df["net_bookings"] = (df.get("bookings", 0) - df.get("cancellations", 0)).clip(lower=0)
    df["conversion_rate_roll6"] = (
        g["retail_sales"].transform(lambda s: s.rolling(6, min_periods=1).sum()) /
        (g["bookings"].transform(lambda s: s.rolling(6, min_periods=1).sum()) + 1)
    )
    df["booking_implied_demand"] = df["net_bookings"] * df["conversion_rate_roll6"]

    df["true_demand"] = (df["retail_sales"] + df["opening_inventory"] - df["closing_inventory"]).clip(lower=0)

    imputed_vals = np.maximum(
        df["booking_implied_demand"],
        g["true_demand"].transform(lambda s: s.rolling(6, min_periods=1).mean())
    )
    df["true_demand"] = np.where(df.get("stockout_flag", 0) == 1, imputed_vals, df["true_demand"])
    df["true_demand"] = df["true_demand"].fillna(0)

    for lag in [1, 3, 6, 12]:
        df[f"demand_lag{lag}"] = g["true_demand"].shift(lag)

    for window in [3, 6]:
        df[f"demand_roll{window}"] = g["true_demand"].shift(1).transform(lambda s: s.rolling(window, min_periods=1).mean())

    df["demand_roll12"] = g["true_demand"].shift(1).transform(lambda s: s.rolling(12, min_periods=6).mean())
    df["demand_roll12_std"] = g["true_demand"].shift(1).transform(lambda s: s.rolling(12, min_periods=6).std()).fillna(0)
    df["yoy_roll_ratio"] = df["demand_roll3"] / (df["demand_roll12"] + 1)
    df["demand_trend3"] = (df["demand_lag1"] - df["demand_lag3"]) / 2.0
    df["yoy_demand_growth"] = (df["demand_lag1"] - df["demand_lag12"]) / (df["demand_lag12"] + 1)

    df["cancel_rate"]  = df.get("cancellations", 0) / (df.get("bookings", 0) + 1)
    df["cancel_rate_roll3"] = g["cancel_rate"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    df["bookings_lag1"] = g["net_bookings"].shift(1)

    df["dealer_inflation"] = df.get("dealer_requested_qty", 0) - df["true_demand"]
    df["dealer_infl_roll3"] = g["dealer_inflation"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    df["panic_order_lag1"] = g["panic_order_flag"].shift(1).fillna(0) if "panic_order_flag" in df.columns else 0

    df["inv_lag1"] = g["closing_inventory"].shift(1)
    df["fill_rate_lag1"] = g["allocated_qty"].shift(1) / (df.get("dealer_requested_qty", 0).shift(1) + 1)
    df["coverage_days"] = df["inv_lag1"] / (df["demand_roll3"] / 30.0 + 0.001)

    if "months_since_launch" in df.columns:
        df["months_since_launch_sq"] = df["months_since_launch"] ** 2
    if "festive_month_flag" in df.columns:
        df["post_festive_trough"] = g["festive_month_flag"].shift(1)

    for k in [1, 2]:
        df[f"sin_month_k{k}"] = np.sin(2 * np.pi * k * df["month"] / 12)
        df[f"cos_month_k{k}"] = np.cos(2 * np.pi * k * df["month"] / 12)

    return df.dropna(subset=["demand_lag12"]).reset_index(drop=True)

def assign_dealer_clusters(train_df: pd.DataFrame, test_df: pd.DataFrame, n_clusters=4):
    cluster_feats = ["demand_roll12", "cancel_rate_roll3", "dealer_infl_roll3", "fill_rate_lag1"]
    avail = [f for f in cluster_feats if f in train_df.columns]

    if not avail:
        train_df["dealer_cluster"] = "Global"
        test_df["dealer_cluster"]  = "Global"
        return train_df, test_df

    profiles = train_df.groupby("dealer_id")[avail].mean().fillna(0).reset_index()
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    profiles["dealer_cluster"] = "Cluster_" + km.fit_predict(profiles[avail]).astype(str)

    cmap = profiles.set_index("dealer_id")["dealer_cluster"].to_dict()
    train_df["dealer_cluster"] = train_df["dealer_id"].map(cmap).fillna("Global")
    test_df["dealer_cluster"]  = test_df["dealer_id"].map(cmap).fillna("Global")

    return train_df, test_df


# ════════════════════════════════════════════════════════════════════════════
# 4. CLUSTER-LEVEL HURDLE + ENSEMBLE ENGINE (WITH VARIANCE GUARDS)
# ════════════════════════════════════════════════════════════════════════════

def _prep_X(X: pd.DataFrame, cat_cols: list[str], for_lgbm: bool = False) -> pd.DataFrame:
    X = X.copy()
    for c in cat_cols:
        if c in X.columns:
            X[c] = X[c].fillna("missing").astype("category" if for_lgbm else str)
    return X

def train_hurdle_ensemble(X_train: pd.DataFrame, y_train: pd.Series, cat_cols: list[str], val_fraction: float = 0.15):
    n_val = max(1, int(len(X_train) * val_fraction))
    X_fit, y_fit = X_train.iloc[:-n_val], y_train.iloc[:-n_val]
    X_val, y_val = X_train.iloc[-n_val:], y_train.iloc[-n_val:]

    y_fit_bin, y_val_bin = (y_fit > 0).astype(int), (y_val > 0).astype(int)

    cat_indices = [i for i, c in enumerate(X_fit.columns) if c in cat_cols]
    classifier = None
    constant_prob = 0.0

    if y_fit_bin.nunique() > 1:
        classifier = CatBoostClassifier(
            iterations=400, learning_rate=0.05, depth=5, loss_function='Logloss',
            early_stopping_rounds=30, verbose=0, cat_features=cat_indices
        )
        classifier.fit(_prep_X(X_fit, cat_cols), y_fit_bin, eval_set=(_prep_X(X_val, cat_cols), y_val_bin), use_best_model=True)
    else:
        constant_prob = float(y_fit_bin.iloc[0]) if not y_fit_bin.empty else 0.0

    mask_fit_nz, mask_val_nz = (y_fit > 0), (y_val > 0)
    cb_regressor, lgb_regressor = None, None

    if mask_fit_nz.sum() > 30:
        cb_regressor = CatBoostRegressor(
            iterations=600, learning_rate=0.03, depth=6, l2_leaf_reg=4.0,
            loss_function='Tweedie:variance_power=1.5', early_stopping_rounds=50, verbose=0, cat_features=cat_indices
        )
        cb_regressor.fit(_prep_X(X_fit[mask_fit_nz], cat_cols), y_fit[mask_fit_nz],
                         eval_set=(_prep_X(X_val[mask_val_nz], cat_cols), y_val[mask_val_nz]), use_best_model=True)

        lgb_train = lgb.Dataset(_prep_X(X_fit[mask_fit_nz], cat_cols, True), label=y_fit[mask_fit_nz])
        lgb_val   = lgb.Dataset(_prep_X(X_val[mask_val_nz], cat_cols, True), label=y_val[mask_val_nz], reference=lgb_train)

        lgb_regressor = lgb.train(
            {'objective': 'tweedie', 'tweedie_variance_power': 1.5, 'metric': 'mae', 'learning_rate': 0.03, 'max_depth': 6, 'verbose': -1},
            lgb_train, num_boost_round=600, valid_sets=[lgb_val], callbacks=[lgb.early_stopping(50, verbose=False)]
        )

    return {"classifier": classifier, "constant_prob": constant_prob, "cb_regressor": cb_regressor, "lgb_regressor": lgb_regressor}


def predict_hurdle_ensemble(models: dict, X_test: pd.DataFrame, cat_cols: list[str]) -> np.ndarray:
    if len(X_test) == 0: return np.array([])

    if models["classifier"] is not None:
        prob_positive = models["classifier"].predict_proba(_prep_X(X_test, cat_cols))[:, 1]
    else:
        prob_positive = np.full(len(X_test), models["constant_prob"])

    if models["cb_regressor"] and models["lgb_regressor"]:
        cb_preds  = np.maximum(0.0, models["cb_regressor"].predict(_prep_X(X_test, cat_cols)))
        lgb_preds = np.maximum(0.0, models["lgb_regressor"].predict(_prep_X(X_test, cat_cols, True)))
        ensemble_vol = (0.5 * cb_preds) + (0.5 * lgb_preds)
    else:
        ensemble_vol = np.zeros(len(X_test))

    return np.maximum(0.0, prob_positive * ensemble_vol)

def train_quantile_model(X_train: pd.DataFrame, y_train: pd.Series, cat_cols: list[str]):
    n_val = max(1, int(len(X_train) * 0.15))
    y_fit = y_train.iloc[:-n_val]

    if y_fit.nunique() <= 1:
        return {"model": None, "constant_val": float(y_fit.iloc[0]) if not y_fit.empty else 0.0}

    model = CatBoostRegressor(
        iterations=500, learning_rate=0.04, depth=6, loss_function="Quantile:alpha=0.85",
        early_stopping_rounds=30, verbose=0, cat_features=[i for i, c in enumerate(X_train.columns) if c in cat_cols]
    )
    model.fit(_prep_X(X_train.iloc[:-n_val], cat_cols), y_fit,
              eval_set=(_prep_X(X_train.iloc[-n_val:], cat_cols), y_train.iloc[-n_val:]), use_best_model=True)
    return {"model": model, "constant_val": 0.0}

def compute_optimal_po(p50: np.ndarray, p85: np.ndarray, horizon: int) -> np.ndarray:
    po = np.minimum(p50 * SAFETY_MULTIPLIERS.get(horizon, 1.12), p85)
    return np.maximum(0, np.round(po)).astype(int)


# ════════════════════════════════════════════════════════════════════════════
# 5. PIPELINE EXECUTION
# ════════════════════════════════════════════════════════════════════════════

def execute_forecast_horizon(df: pd.DataFrame, test_month: str, horizon: int, is_validation: bool = False):
    target_col = f"target_h{horizon}"
    df[target_col] = df.groupby(["dealer_id", "model", "variant"])["true_demand"].shift(-(horizon - 1))

    train_cutoff_month = offset_month(test_month, -horizon)
    train_df = df[(df["year_month"] <= train_cutoff_month) & df[target_col].notna()].copy()
    test_df  = df[df["year_month"] == test_month].copy()

    if train_df.empty or test_df.empty:
        return pd.DataFrame()

    train_df, test_df = assign_dealer_clusters(train_df, test_df)

    hist_len  = train_df.groupby(["dealer_id", "model", "variant"])["year_month"].nunique()
    thin_keys = set(hist_len[hist_len < MIN_HISTORY_MONTHS].index)
    test_suf  = test_df[~test_df.set_index(["dealer_id", "model", "variant"]).index.isin(thin_keys)].copy()

    feats_avail = [f for f in FEATURES if f in df.columns]
    cat_avail   = [c for c in CAT_FEATURES if c in feats_avail]

    results = []

    if not test_suf.empty:
        suf_out = test_suf[["dealer_id", "model", "variant", "true_demand"]].copy()
        suf_out["predicted_demand"] = 0
        suf_out["po_upper_bound"] = 0

        for cluster in train_df["dealer_cluster"].unique():
            tr_c = train_df[train_df["dealer_cluster"] == cluster]
            te_c = test_suf[test_suf["dealer_cluster"] == cluster]

            if len(te_c) == 0 or len(tr_c) < 100: continue

            cluster_models = train_hurdle_ensemble(tr_c[feats_avail], tr_c[target_col], cat_avail)
            cluster_q85    = train_quantile_model(tr_c[feats_avail], tr_c[target_col], cat_avail)

            p50 = predict_hurdle_ensemble(cluster_models, te_c[feats_avail], cat_avail)

            if cluster_q85["model"] is not None:
                p85 = np.maximum(0.0, cluster_q85["model"].predict(_prep_X(te_c[feats_avail], cat_avail)))
            else:
                p85 = np.full(len(te_c), cluster_q85["constant_val"])

            suf_out.loc[te_c.index, "predicted_demand"] = p50
            suf_out.loc[te_c.index, "po_upper_bound"]   = p85

        suf_out["optimal_po"] = compute_optimal_po(suf_out["predicted_demand"].values, suf_out["po_upper_bound"].values, horizon)
        suf_out["predicted_demand"] = np.round(suf_out["predicted_demand"]).astype(int)
        suf_out["po_upper_bound"]   = np.round(suf_out["po_upper_bound"]).astype(int)

        if is_validation:
            target = df[df["year_month"] == offset_month(test_month, horizon - 1)][["dealer_id", "model", "variant", "true_demand"]].rename(columns={"true_demand": "actual"})
            suf_out = suf_out.merge(target, on=["dealer_id", "model", "variant"], how="inner")

        results.append(suf_out)

    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()


def run_pipeline(run_validation=True):
    print("Loading and preparing data...")
    df = load_and_merge()
    if df.empty: return

    df = build_features(df)
    all_months = sorted(df["year_month"].unique())

    if run_validation:
        print("\n── Strict Gap Validation (Hurdle Ensemble + Clusters) ──")
        for H in HORIZONS:
            test_months = all_months[-(VALIDATION_WINDOWS + H):-H]
            wapes = []

            for tm in test_months:
                res = execute_forecast_horizon(df.copy(), tm, H, is_validation=True)
                if not res.empty and "actual" in res.columns:
                    w = np.sum(np.abs(res["predicted_demand"] - res["actual"])) / (np.sum(res["actual"]) + 1e-9) * 100
                    wapes.append(w)

            if wapes:
                print(f"  Horizon {H}M | Mean WAPE: {np.mean(wapes):.1f}% ± {np.std(wapes):.1f}pp")
            else:
                print(f"  Horizon {H}M | Insufficient historical data for {VALIDATION_WINDOWS} windows.")

    print("\n── Generating Final Production Forecasts ──")
    final_month = all_months[-1]
    all_results = []

    for H in HORIZONS:
        print(f"\n── Horizon H={H}M ──────────────────────────────────────")
        res = execute_forecast_horizon(df.copy(), final_month, H, is_validation=False)

        if not res.empty:
            res["horizon_months"] = H
            res["forecast_origin_month"] = final_month
            res["target_forecast_month"] = offset_month(final_month, H)
            all_results.append(res)

            # ── ORIGINAL METRICS PRINT RESTORED ──
            if H == 1 and "true_demand" in res.columns:
                print("  Metrics (H=1 only, ground truth available):")
                y_true = res["true_demand"].to_numpy()
                y_pred = res["predicted_demand"].to_numpy()
                y_po   = res["optimal_po"].to_numpy()
                wape = np.sum(np.abs(y_pred - y_true)) / (np.sum(y_true) + 1e-9) * 100
                fill_rate = (y_po >= y_true).mean() * 100
                print(f"  WAPE: {wape:.1f}% | Fill rate: {fill_rate:.1f}%")
            elif H > 1:
                print(f"  Predictions generated for Month +{H}.")

    if all_results:
        final_df = pd.concat(all_results, ignore_index=True)
        final_df.to_csv("dealer_demand_forecast_v2.csv", index=False)

        # ── RESTORED: Aggregate Production Plan ──
        plan = final_df.groupby(["model", "variant", "target_forecast_month", "horizon_months"]).agg(
            total_predicted_demand=("predicted_demand", "sum"),
            total_optimal_po=("optimal_po", "sum")
        ).reset_index()
        plan.to_csv("monthly_production_plan_v2.csv", index=False)

        # ── RESTORED: Regional Allocation Plan ──
        # Merge region/zone info back if it was dropped during test_suf slicing
        if "region" not in final_df.columns:
            final_df = final_df.merge(df[["dealer_id", "region", "zone"]].drop_duplicates(), on="dealer_id", how="left")

        regional = final_df.groupby(["region", "zone", "model", "variant", "target_forecast_month", "horizon_months"]).agg(
            regional_predicted=("predicted_demand", "sum"),
            regional_po=("optimal_po", "sum")
        ).reset_index()
        regional.to_csv("regional_allocation_plan_v2.csv", index=False)

        print("\n── Outputs saved ───────────────────────────────────────────────")
        print("1. dealer_demand_forecast_v2.csv (Granular)")
        print("2. monthly_production_plan_v2.csv (Factory Roll-up)")
        print("3. regional_allocation_plan_v2.csv (Zone Roll-up)")

if __name__ == "__main__":
    run_pipeline()

Loading and preparing data...

── Strict Gap Validation (Hurdle Ensemble + Clusters) ──
  Horizon 1M | Mean WAPE: 17.4% ± 0.0pp
  Horizon 3M | Mean WAPE: 29.8% ± 0.0pp
  Horizon 6M | Mean WAPE: 29.9% ± 0.0pp
  Horizon 12M | Mean WAPE: 29.9% ± 0.0pp

── Generating Final Production Forecasts ──

── Horizon H=1M ──────────────────────────────────────
  Metrics (H=1 only, ground truth available):
  WAPE: 20.5% | Fill rate: 74.6%

── Horizon H=3M ──────────────────────────────────────
  Predictions generated for Month +3.

── Horizon H=6M ──────────────────────────────────────
  Predictions generated for Month +6.

── Horizon H=12M ──────────────────────────────────────
  Predictions generated for Month +12.

── Outputs saved ───────────────────────────────────────────────
1. dealer_demand_forecast_v2.csv (Granular)
2. monthly_production_plan_v2.csv (Factory Roll-up)
3. regional_allocation_plan_v2.csv (Zone Roll-up)
